In [0]:
spark

In [0]:
%sql
create table sales_input(
store_id int,
sale_date date,
total_sales int
);


In [0]:
%sql
INSERT INTO sales_input (store_id, sale_date, total_sales) VALUES
(101, '2024-01-01', 500),
(101, '2024-01-02', NULL),
(101, '2024-01-03', NULL),
(101, '2024-01-04', 700),
(101, '2024-01-05', NULL);

In [0]:
%sql
select * from sales_input

# Approach 1(using two lag functions and using case when to filter off the nulls)

In [0]:
%sql

with lagged_sales as (
select A.*,
lag(new_total_sales) OVER(order by A.sale_date ASC) as laged_total_sales
from (
select store_id,
sale_date,
total_sales,
lag(total_sales) over(order by sale_date asc) as new_total_sales
from sales_input 
ORDER BY sale_date asc
) A
)
select store_id,
sale_date,
case when total_sales is not null then total_sales
when new_total_sales is not null then new_total_sales
when laged_total_sales is not null then laged_total_sales
end as  total_sales
FROM 
lagged_sales

# Approach two using Lag with Ingore Nulls and case when to filter off the nulls

In [0]:
%sql
explain 

select A.store_id,
A.sale_date,
case when A.total_sales is not null then total_sales
when A.new_total_sales is not null then new_total_sales
end as  total_sales
from (
select store_id,
sale_date,
total_sales,
lag(total_sales,1) ignore nulls  over(partition by store_id order by sale_date) as new_total_sales
from sales_input
) A


# Approach 3 using Coalesce long with Lag and Ignore nulls 

In [0]:
%sql
explain 

select store_id,
sale_date,
total_sales,
COALESCE(total_sales, lag(total_sales) ignore nulls  over(partition by store_id order by sale_date)) as total_sale
from sales_input

# Create a team match table based on the data given in the players table with the conditions as below
1. Eeach player in the list should play with all other players (one to one).
2. A player cannot play with himself/herself.
3. The combination should be like p1 vs p2 but not the other way around since p1 vs p2 is same as p2 vs p1

In [0]:
%sql
create table players (
  player_name string
);

insert into players (player_name) values
("p1"),
("p2"),
("p3"),
("p4"),
("p5");

select * from players

In [0]:
%sql
select 
A.player_name ,
B.player_name
from players as A 
cross join players as B 
where A.player_name != B.player_name 
and A.player_name < B.player_name
order by A.player_name, B.player_name

You're given two tables containing data on Spotify users' streaming activity: songs_history which has historical streaming data, and songs_weekly which has data from the current week.

Write a query that outputs the user ID, song ID, and cumulative count of song plays up to August 4th, 2022, sorted in descending order.

Assume that there may be new users or songs in the songs_weekly table that are not present in the songs_history table.

Definitions:

song_weeklytable only contains data for the week of August 1st to August 7th, 2022.
songs_history table contains data up to July 31st, 2022. The query should include historical data from this table.
songs_history Table:
Column Name	Type
history_id	integer
user_id	integer
song_id	integer
song_plays	integer
songs_history Example Input:
history_id	user_id	song_id	song_plays
10011	777	1238	11
12452	695	4520	1
song_plays field contains the historical data of the number of times a user has played a particular song.

songs_weekly Table:
Column Name	Type
user_id	integer
song_id	integer
listen_time	datetime
songs_weekly Example Input:
user_id	song_id	listen_time
777	1238	08/01/2022 12:00:00
695	4520	08/04/2022 08:00:00
125	9630	08/04/2022 16:00:00
695	9852	08/07/2022 12:00:00
Example Output:
user_id	song_id	song_plays
777	1238	12
695	4520	2
125	9630	1
On 4 August 2022, the data shows that User 777 listened to the song with song ID 1238 for a total of 12 times, with 11 of those times occurring before the current week and 1 time occurring within the current week.

However, the streaming data for User 695 with the song ID 9852 are not included in the output because the streaming date for that record falls outside the date range specified in the question.

The dataset you are querying against may have different input & output - this is just an example!

In [0]:
%sql
CREATE TABLE songs_history (
  history_id INT,
  user_id INT,
  song_id INT,
  song_plays INT
);

INSERT INTO songs_history (history_id, user_id, song_id, song_plays) VALUES
(10011, 777, 1238, 11),
(12452, 695, 4520, 1);

CREATE TABLE songs_weekly (
  user_id INT,
  song_id INT,
  listen_time TIMESTAMP
);

INSERT INTO songs_weekly (user_id, song_id, listen_time) VALUES
(777, 1238, '2022-08-01 12:00:00'),
(695, 4520, '2022-08-04 08:00:00'),
(125, 9630, '2022-08-04 16:00:00'),
(695, 9852, '2022-08-07 12:00:00');

    
select * from songs_weekly;

In [0]:
%sql
select * from songs_history;

In [0]:
%sql
with weekly_songs as (
select user_id, 
song_id 
from songs_weekly weekly
where to_date(weekly.listen_time) = '2022-08-07'
),
hist_songs as (
select user_id,
song_id
from songs_history
)
select user_id, song_id
from weekly_songs
union all
select user_id, song_id
from hist_songs

In [0]:
%sql
select user_id, 
song_id,
count(song_id) over(partition by user_id) as weekly_count
from songs_weekly

In [0]:
%sql
with total_songs as (
select 
user_id,
song_id,
1 as total_plays
from songs_weekly
where TO_DATE(listen_time) <= '2022-08-04'
union all
select user_id,
song_id,
song_plays as total_plays
from songs_history
)
select user_id,
song_id,
SUM(total_plays) as total_plays
from total_songs
GROUP BY user_id,
song_id
order by total_plays desc


This is the same question as problem #28 in the SQL Chapter of Ace the Data Science Interview!

Assume you're given a table with measurement values obtained from a Google sensor over multiple days with measurements taken multiple times within each day.

Write a query to calculate the sum of odd-numbered and even-numbered measurements separately for a particular day and display the results in two different columns. Refer to the Example Output below for the desired format.

Definition:

Within a day, measurements taken at 1st, 3rd, and 5th times are considered odd-numbered measurements, and measurements taken at 2nd, 4th, and 6th times are considered even-numbered measurements.
Effective April 15th, 2023, the question and solution for this question have been revised.

measurements Table:
Column Name	Type
measurement_id	integer
measurement_value	decimal
measurement_time	datetime
measurements Example Input:
measurement_id	measurement_value	measurement_time
131233	1109.51	07/10/2022 09:00:00
135211	1662.74	07/10/2022 11:00:00
523542	1246.24	07/10/2022 13:15:00
143562	1124.50	07/11/2022 15:00:00
346462	1234.14	07/11/2022 16:45:00
Example Output:
measurement_day	odd_sum	even_sum
07/10/2022 00:00:00	2355.75	1662.74
07/11/2022 00:00:00	1124.50	1234.14
Explanation
Based on the results,

On 07/10/2022, the sum of the odd-numbered measurements is 2355.75, while the sum of the even-numbered measurements is 1662.74.
On 07/11/2022, there are only two measurements available. The sum of the odd-numbered measurements is 1124.50, and the sum of the even-numbered measurements is 1234.14.
The dataset you are querying against may have different input & output - this is just an example!

expected output:
| measurement_day    | odd_sum | even_sum |
|--------------------|---------|----------|
| 07/10/2022 00:00:00 | 2355.75 | 1662.74  |
| 07/11/2022 00:00:00 | 1124.50 | 1234.14  |

In [0]:
%sql
CREATE TABLE measurements (
    measurement_id INT,
    measurement_value DECIMAL(10, 2), -- Assuming up to 10 digits total, 2 after the decimal point
    measurement_time TIMESTAMP
);

INSERT INTO measurements (measurement_id, measurement_value, measurement_time) VALUES
(131233, 1109.51, '2022-07-10 09:00:00'),
(135211, 1662.74, '2022-07-10 11:00:00'),
(523542, 1246.24, '2022-07-10 13:15:00'),
(143562, 1124.50, '2022-07-11 15:00:00'),
(346462, 1234.14, '2022-07-11 16:45:00');


select * from measurements;

In [0]:
%sql
select 
timemeasurement_time as measurement_day

SQL joins question

In [0]:
%sql
create table TableA (
  colA int
);

insert into TableA (colA) values
(1),
(1),
(1),
(2),
(2),
(3),
(5)



In [0]:
%sql
select * from TableA